# PARÁMETROS DEL CATÁLOGO

In [0]:
dbutils.widgets.text("catalog", "electrocasa_dev", "Catálogo")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")
print(f"Aplicando gobierno en catálogo: {catalog}")

# GRANT - 3 GRUPOS

In [0]:
# 1. electrocasa_ingenieria: Lectura y escritura sobre bronze, silver y gold
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `electrocasa_ingenieria`")
for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"GRANT ALL PRIVILEGES ON SCHEMA {catalog}.{schema} TO `electrocasa_ingenieria`")

# 2. electrocasa_analistas: Solo lectura sobre gold[cite: 3]
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `electrocasa_analistas`")
spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {catalog}.gold TO `electrocasa_analistas`")

# 3. electrocasa_auditoria: Solo lectura sobre gold + evidencia de auditoría y lineage[cite: 3]
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `electrocasa_auditoria`")
spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {catalog}.gold TO `electrocasa_auditoria`")

print("Permisos por grupo aplicados correctamente.")

# ACCESO A AUDITORÍA

In [0]:
try:
    spark.sql("GRANT USE CATALOG ON CATALOG system TO `electrocasa_auditoria`")
    spark.sql("GRANT USE SCHEMA ON SCHEMA system.access TO `electrocasa_auditoria`")
    spark.sql("GRANT SELECT ON TABLE system.access.audit TO `electrocasa_auditoria`")
    spark.sql("GRANT SELECT ON TABLE system.access.table_lineage TO `electrocasa_auditoria`")
    spark.sql("GRANT SELECT ON TABLE system.access.column_lineage TO `electrocasa_auditoria`")
    print("Permisos sobre system.access otorgados correctamente.")
except Exception as e:
    print("Error al otorgar permisos en catálogo system (guardar este mensaje para el README):")
    print(e)

# REVOKES

In [0]:
print("--- 1. Otorgando privilegio indebido (SELECT en silver a electrocasa_analistas) ---")
spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {catalog}.silver TO `electrocasa_analistas`")

print("--- SHOW GRANTS ANTES DEL REVOKE ---")
display(spark.sql(f"SHOW GRANTS ON SCHEMA {catalog}.silver"))

print("--- 2. Revocando privilegio indebido ---")
spark.sql(f"REVOKE USE SCHEMA, SELECT ON SCHEMA {catalog}.silver FROM `electrocasa_analistas`")

print("--- SHOW GRANTS DESPUÉS DEL REVOKE ---")
display(spark.sql(f"SHOW GRANTS ON SCHEMA {catalog}.silver"))